Lets begin by reading data/seq.txt into a list of sequences we can use

In [2]:

seqs = []
with open('data/seq.txt') as f:
    for line in f:
        line = line.strip()
        # Strip the last 3 characters 'DNA'
        seqs.append(line[:-3])
        seqs.append(line.strip())

print(seqs)

['...']


Lets begin by creating the comparison matrix between all sequences, using BLOOM62

In [3]:
from Bio.Align import PairwiseAligner
from Bio.Align import substitution_matrices
import numpy as np

def compute_smith_waterman_bioaligner(seq1, seq2):
    """
    Computes the Smith-Waterman alignment score for two sequences using PairwiseAligner.
    """
    # Load BLOSUM62 substitution matrix
    matrix = substitution_matrices.load("BLOSUM62")

    # Initialize the aligner
    aligner = PairwiseAligner()
    aligner.substitution_matrix = matrix
    aligner.open_gap_score = -4
    aligner.extend_gap_score = -1

    # Perform the alignment
    score = aligner.score(seq1, seq2)
    return score


def compute_comparison_matrix(sequences):
    """
    Computes a symmetric comparison matrix for a list of sequences.
    """
    n = len(sequences)
    comparison_matrix = np.zeros((n, n))

    print("Starting comparison matrix computation...")
    for i in range(n):
        print(f"Processing sequence {i+1}/{n}")
        for j in range(i, n):
            score = compute_smith_waterman_bioaligner(sequences[i], sequences[j])
            comparison_matrix[i, j] = score
            comparison_matrix[j, i] = score  # Symmetry

    print("Finished computation.")
    return comparison_matrix

Now lets begin by creating a comparison matrix between all the sequences and displaying it nicely

In [9]:
original_matrix = compute_comparison_matrix(seqs)
print(original_matrix)

Starting comparison matrix computation...
Processing sequence 1/272
Processing sequence 2/272
Processing sequence 3/272
Processing sequence 4/272
Processing sequence 5/272
Processing sequence 6/272
Processing sequence 7/272
Processing sequence 8/272
Processing sequence 9/272
Processing sequence 10/272
Processing sequence 11/272
Processing sequence 12/272
Processing sequence 13/272
Processing sequence 14/272
Processing sequence 15/272
Processing sequence 16/272
Processing sequence 17/272
Processing sequence 18/272
Processing sequence 19/272
Processing sequence 20/272
Processing sequence 21/272
Processing sequence 22/272
Processing sequence 23/272
Processing sequence 24/272
Processing sequence 25/272
Processing sequence 26/272
Processing sequence 27/272
Processing sequence 28/272
Processing sequence 29/272
Processing sequence 30/272
Processing sequence 31/272
Processing sequence 32/272
Processing sequence 33/272
Processing sequence 34/272
Processing sequence 35/272
Processing sequence 36

Note how a higher allignment score means the sequences are more closely matched. 
Now, lets build out the tree, beginning with a matrix of all sequences, and gradually "building up", keeping track of the merges that we made.

In [49]:
import numpy as np
from copy import deepcopy

# Initialize clusters and Newick strings
current_clusters = [[i] for i in range(len(seqs))]
newick_nodes = [str(i) for i in range(len(seqs))]

def compute_average_score(cluster1, cluster2, original_matrix, linkage='average'):
    scores = [original_matrix[elem1, elem2] for elem1 in cluster1 for elem2 in cluster2]
    if linkage == 'average':
        return np.mean(scores)
    elif linkage == 'complete':
        return np.max(scores)
    elif linkage == 'single':
        return np.min(scores)


while len(current_clusters) > 1:
    # Find the highest normalized score
    max_score = -np.inf
    to_merge = (None, None)
    
    for i in range(len(current_clusters)):
        for j in range(i + 1, len(current_clusters)):
            # Use average linkage
            score = compute_average_score(current_clusters[i], current_clusters[j], original_matrix)
            if score > max_score:
                max_score = score
                to_merge = (i, j)

    # Merge the clusters with the highest score
    cluster1_idx, cluster2_idx = to_merge
    cluster1 = current_clusters[cluster1_idx]
    cluster2 = current_clusters[cluster2_idx]
    new_cluster = cluster1 + cluster2

    # Create new Newick node with distances
    new_newick = f"({newick_nodes[cluster1_idx]},{newick_nodes[cluster2_idx]})"
    
    # Update newick_nodes list
    newick_nodes = (
        newick_nodes[:cluster1_idx] + 
        [new_newick] + 
        newick_nodes[cluster1_idx + 1:cluster2_idx] + 
        newick_nodes[cluster2_idx + 1:]
    )
    
    # Note in the above, how the newick nodes indexes will match the current_clusters indexes

    # Update the clusters
    current_clusters = (
        current_clusters[:cluster1_idx] + 
        [new_cluster] + 
        current_clusters[cluster1_idx + 1:cluster2_idx] + 
        current_clusters[cluster2_idx + 1:]
    )

    # Print progress
    print(f"Merged clusters {cluster1_idx} and {cluster2_idx} with score {max_score:.4f}")
    print(f"Current clusters: {current_clusters}")

# Add final semicolon to complete Newick string
final_newick = newick_nodes[0] + ";"
print("\nFinal Newick string:", final_newick)

Merged clusters 90 and 91 with score 2861.0000
Current clusters: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17], [18], [19], [20], [21], [22], [23], [24], [25], [26], [27], [28], [29], [30], [31], [32], [33], [34], [35], [36], [37], [38], [39], [40], [41], [42], [43], [44], [45], [46], [47], [48], [49], [50], [51], [52], [53], [54], [55], [56], [57], [58], [59], [60], [61], [62], [63], [64], [65], [66], [67], [68], [69], [70], [71], [72], [73], [74], [75], [76], [77], [78], [79], [80], [81], [82], [83], [84], [85], [86], [87], [88], [89], [90, 91], [92], [93], [94], [95], [96], [97], [98], [99], [100], [101], [102], [103], [104], [105], [106], [107], [108], [109], [110], [111], [112], [113], [114], [115], [116], [117], [118], [119], [120], [121], [122], [123], [124], [125], [126], [127], [128], [129], [130], [131], [132], [133], [134], [135], [136], [137], [138], [139], [140], [141], [142], [143], [144], [145], [146], [147], [148], [14

Now, let's display the tree nicely

In [50]:
from ete3 import Tree

tree = Tree(final_newick)
print(tree)


                                                                                                                                                                                                                                                                                                                                                                                                                  /-0
                                                                                                                                                                                                                                                                                                                                                                                                               /-|
                                                                                                                                                                                              